# SmartBite v3 — Persistent Colab Pipeline

This replaces the old audit notebook. Dataset downloads, manifests, reports and deduplication metadata are stored in Google Drive so a Colab runtime reset does not erase the source data or completed audit work. Temporary resized training images remain in `/content` and can be rebuilt.

## Before running
1. Open this notebook in Google Colab and select a GPU runtime (T4 is sufficient for the embedding pass).
2. In Colab **Secrets**, add `KAGGLE_USERNAME` and `KAGGLE_KEY`; enable notebook access.
3. Run cells from top to bottom. Do not run cells out of order.
4. The notebook stops after dataset preparation and packages a review ZIP. **Training is deliberately gated** until class counts, label conflicts and visual samples are reviewed. Set `RUN_TRAINING = True` only after approval.

Khana is optional because the current config has no public Kaggle reference for it. The audit will report it as missing rather than fail the entire pipeline.

In [ ]:
# 0. Mount Drive, install only the packages this repository needs, and restore the branch.
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, json, csv, shutil, subprocess, zipfile, hashlib, time
from pathlib import Path
import yaml

REPO = Path('/content/cv_model')
BRANCH = 'taxonomy-v3-audit'
REPO_URL = 'https://github.com/ashraff-56/cv_model.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/SmartBite/v3_pipeline')
DATA_HOME = DRIVE_ROOT / 'datasets'          # persistent source datasets
STATE_HOME = DRIVE_ROOT / 'state'            # persistent caches and config
REPORT_HOME = DRIVE_ROOT / 'reports'         # persistent reports and manifest
for p in (DATA_HOME, STATE_HOME, REPORT_HOME): p.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True, env=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    print(result.stdout[-12000:] if result.stdout else '')
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed ({result.returncode}): {cmd[0]}')
    return result

if not (REPO / '.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO)])
else:
    run(['git','fetch','origin',BRANCH], cwd=REPO)
    run(['git','checkout',BRANCH], cwd=REPO)
    run(['git','pull','--ff-only','origin',BRANCH], cwd=REPO)

# Do not reinstall torch/torchvision: Colab supplies compatible GPU builds.
run([sys.executable,'-m','pip','install','-q','kaggle','pyyaml','pillow','timm','pytest'])

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')
if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise RuntimeError('Missing Colab Secrets. Add KAGGLE_USERNAME and KAGGLE_KEY, enable notebook access, then rerun this cell.')
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY
os.environ['KAGGLE_CONFIG_DIR'] = '/content'

CFG_PATH = REPO / 'configs/data_v3.yaml'
with CFG_PATH.open(encoding='utf-8') as f: BASE_CFG = yaml.safe_load(f)
print('\nRepository:', run(['git','rev-parse','--short','HEAD'],cwd=REPO).stdout.strip())
print('Persistent Drive root:', DRIVE_ROOT)
print('GPU available:', run([sys.executable,'-c','import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")'],check=True).stdout.strip())


## 1. Download or reuse datasets

Completed datasets already extracted in Drive are reused. New datasets download as one ZIP to Colab local storage, are integrity-checked, backed up as a single ZIP in Drive, and extracted locally with progress. This avoids extracting thousands of small files directly into mounted Drive. After a runtime reset, the ZIP is restored and extracted locally.


In [ ]:
def find_marker(root: Path, marker: str, max_depth=9):
    """Find a configured dataset marker without traversing arbitrarily deep trees."""
    root = Path(root)
    if not root.exists():
        return None
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            continue
        if depth > max_depth:
            dirs[:] = []
            continue
        if (current_path / marker).exists():
            return current_path
    return None

def prepare_food5k_layout(root: Path):
    """Normalize Food-5K's 0_/1_ filename labels into split/class folders."""
    root = Path(root)
    if not root.exists():
        return None
    normalized = root / "_smartbite_food5k_normalized"

    def image_count(folder):
        return sum(1 for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")) if folder.is_dir() else 0

    # Find the original split directories, including an optional enclosing Food-5K folder.
    split_dirs = []
    for p in root.rglob("*"):
        if not p.is_dir() or "_smartbite_food5k_normalized" in p.parts:
            continue
        if p.name.lower() in ("training", "validation", "evaluation"):
            split_dirs.append(p)

    copied = 0
    for split_dir in split_dirs:
        split = split_dir.name.lower()
        for src in split_dir.iterdir():
            if not src.is_file() or src.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
                continue
            prefix = src.stem.split("_", 1)[0]
            label = {"0": "non_food", "1": "food"}.get(prefix)
            if label is None:
                continue
            dest_dir = normalized / split / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest = dest_dir / src.name
            if not dest.exists():
                shutil.copy2(src, dest)
                copied += 1

    # The source config needs training/non_food; require both training labels to
    # avoid silently treating an incomplete extraction as a usable dataset.
    train_food = image_count(normalized / "training" / "food")
    train_non_food = image_count(normalized / "training" / "non_food")
    if train_food and train_non_food:
        print(f"Food-5K normalized: {copied:,} images; training food={train_food:,}, non_food={train_non_food:,}", flush=True)
        return normalized

    observed = {p.name: sorted(q.name for q in p.iterdir() if q.is_dir())
                for p in split_dirs}
    counts = {f"{split}/{label}": image_count(normalized / split / label)
              for split in ("training", "validation", "evaluation")
              for label in ("food", "non_food")}
    raise RuntimeError(
        f"Food-5K files were extracted but could not build training/food and training/non_food. "
        f"Split folders found: {observed}; normalized image counts: {counts}. "
        "Expected flat image files named 0_<id>.jpg (non_food) and 1_<id>.jpg (food)."
    )


def download_kaggle_with_retries(ref: str, destination: Path, attempts: int = 3):
    """Download into an isolated folder; retry transient failures and never select an old ZIP."""
    destination.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, attempts + 1):
        print(f'Kaggle download attempt {attempt}/{attempts}: {ref}', flush=True)
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', ref, '-p', str(destination), '-o'],
            cwd='/content', text=True, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, check=False
        )
        if result.stdout:
            print(result.stdout[-8000:], flush=True)
        zips = sorted(destination.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
        if result.returncode == 0 and zips:
            candidate = zips[0]
            try:
                with zipfile.ZipFile(candidate) as archive_file:
                    bad = archive_file.testzip()
                    if bad:
                        raise RuntimeError(f'Corrupt ZIP member: {bad}')
                    print(f'ZIP integrity OK: {len(archive_file.infolist()):,} entries', flush=True)
                return candidate
            except Exception as exc:
                last_error = f'ZIP validation failed: {type(exc).__name__}: {exc}'
        else:
            last_error = f'Kaggle exit code {result.returncode}; no ZIP produced'
        if attempt < attempts:
            wait_seconds = 5 * (2 ** (attempt - 1))
            print(f'{last_error}; retrying in {wait_seconds}s', flush=True)
            time.sleep(wait_seconds)
    raise RuntimeError(f'Kaggle download failed after {attempts} attempts: {last_error}')

# Download Kaggle ZIPs to Colab local disk; never unzip thousands of files directly into mounted Drive.
# Existing completed Drive datasets are reused. Incomplete folders are left untouched.
LOCAL_DATA_HOME = Path('/content/smartbite_v3_datasets')
LOCAL_ZIP_HOME = Path('/content/kaggle_zips')
ARCHIVE_HOME = DATA_HOME / '_archives'
for p in (LOCAL_DATA_HOME, LOCAL_ZIP_HOME, ARCHIVE_HOME):
    p.mkdir(parents=True, exist_ok=True)

def extract_zip_with_progress(zip_path: Path, target: Path):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f'Corrupt ZIP member: {bad}')
        members = z.infolist()
        total = len(members)
        print(f'ZIP verified: {total:,} entries; extracting to {target}', flush=True)
        for i, member in enumerate(members, 1):
            z.extract(member, target)
            if i % 2000 == 0 or i == total:
                print(f'  Extracted {i:,}/{total:,} entries', flush=True)
    return find_marker(target, marker)

DOWNLOAD_LOG = []
RUNTIME_CFG = yaml.safe_load(yaml.safe_dump(BASE_CFG))

for ds_id, spec in RUNTIME_CFG['datasets'].items():
    try:
        ref = spec.get('kaggle_ref')
        marker = spec['marker']
        drive_dest = DATA_HOME / ds_id
        local_dest = LOCAL_DATA_HOME / ds_id
        drive_dest.mkdir(parents=True, exist_ok=True)
        local_dest.mkdir(parents=True, exist_ok=True)

        # Food-5K stores labels in filename prefixes (0=non-food, 1=food), not class folders.
        # Normalize it before marker checks so the configured */non_food source glob works.
        if ds_id == 'food5k':
            prepare_food5k_layout(drive_dest)
            prepare_food5k_layout(local_dest)

        # Reuse a fully extracted dataset already present in Drive.
        existing_drive = find_marker(drive_dest, marker)
        if existing_drive:
            spec['local'] = str(existing_drive)
            DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'REUSED_FROM_DRIVE', 'root': str(existing_drive), 'ref': ref})
            print('REUSED', ds_id, '->', existing_drive, flush=True)
            continue

        # If this runtime already has a complete local extraction, reuse it.
        existing_local = find_marker(local_dest, marker)
        if existing_local:
            spec['local'] = str(existing_local)
            DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'REUSED_LOCAL', 'root': str(existing_local), 'ref': ref})
            print('REUSED LOCAL', ds_id, '->', existing_local, flush=True)
            continue

        archive = ARCHIVE_HOME / f'{ds_id}.zip'
        # After a runtime reset, restore from the persistent ZIP in Drive.
        if archive.exists() and archive.stat().st_size > 0:
            print(f'RESTORING {ds_id} from Drive archive ({archive.stat().st_size / 1e9:.2f} GB)', flush=True)
            extract_zip_with_progress(archive, local_dest)
            if ds_id == 'food5k':
                found = prepare_food5k_layout(local_dest)
            else:
                found = find_marker(local_dest, marker)
            if found:
                spec['local'] = str(found)
                DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'RESTORED_FROM_DRIVE_ZIP', 'root': str(found), 'ref': ref})
                print('READY', ds_id, '->', found, flush=True)
                continue
            raise RuntimeError(f'Archive extracted but expected marker {marker!r} was not found for {ds_id}')

        if not ref:
            DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'OPTIONAL_SOURCE_NO_PUBLIC_KAGGLE_REF'})
            print('SKIPPED (no Kaggle ref):', ds_id, flush=True)
            continue

        print(f'\nDownloading {ds_id} from {ref} to Colab local storage', flush=True)
        print('This downloads one ZIP first; extraction happens locally with progress.', flush=True)
        started = time.time()
        dataset_zip_dir = LOCAL_ZIP_HOME / ds_id
        downloaded_zip = download_kaggle_with_retries(ref, dataset_zip_dir, attempts=3)
        elapsed = round(time.time() - started, 1)
        print(f'Download complete: {downloaded_zip.name} ({downloaded_zip.stat().st_size / 1e9:.2f} GB)', flush=True)
        with zipfile.ZipFile(downloaded_zip) as z:
            bad = z.testzip()
            if bad:
                raise RuntimeError(f'Downloaded ZIP is corrupt at member: {bad}')
            print(f'ZIP integrity OK: {len(z.infolist()):,} entries', flush=True)

        # Persist the single archive to Drive; do not create thousands of Drive files.
        archive_tmp = archive.with_suffix('.zip.partial')
        print(f'Backing up archive to Drive: {archive}', flush=True)
        shutil.copy2(downloaded_zip, archive_tmp)
        archive_tmp.replace(archive)
        print('Drive archive backup complete.', flush=True)

        extract_zip_with_progress(downloaded_zip, local_dest)
        if ds_id == 'food5k':
            found = prepare_food5k_layout(local_dest)
        else:
            found = find_marker(local_dest, marker)
        if not found:
            raise RuntimeError(f'{ds_id} extracted but marker {marker!r} was not found. Check dataset structure.')
        spec['local'] = str(found)
        DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'DOWNLOADED_AND_ARCHIVED',
                             'root': str(found), 'archive': str(archive), 'ref': ref,
                             'elapsed_seconds': elapsed})
        print('READY', ds_id, '->', found, flush=True)


    except Exception as exc:
        error_text = f'{type(exc).__name__}: {exc}'
        DOWNLOAD_LOG.append({'dataset': ds_id, 'status': 'FAILED', 'ref': ref, 'error': error_text})
        print(f'\nFAILED dataset {ds_id}: {error_text}', flush=True)
        print('Continuing with the next dataset. See the persistent error log.', flush=True)
    finally:
        log_tmp = STATE_HOME / 'download_log.json.tmp'
        log_tmp.write_text(json.dumps(DOWNLOAD_LOG, indent=2), encoding='utf-8')
        log_tmp.replace(STATE_HOME / 'download_log.json')
# Khana remains optional; configure a real Kaggle slug only if one is available.
KHANA_KAGGLE_REF = ''
if KHANA_KAGGLE_REF.strip():
    print('Khana custom source is not auto-configured in this pipeline; add it as a dataset entry first.', flush=True)

print('\nDOWNLOAD STATUS')
for row in DOWNLOAD_LOG:
    print(row)
print('\nPersistent Drive datasets:', DATA_HOME)
print('Persistent ZIP archives:', ARCHIVE_HOME)
print('Local extracted datasets:', LOCAL_DATA_HOME)

failed_rows = [row for row in DOWNLOAD_LOG if row.get('status') == 'FAILED']
print(f'\nDataset summary: {len(DOWNLOAD_LOG) - len(failed_rows)} completed/skipped, {len(failed_rows)} failed')
if failed_rows:
    print('Failed datasets:')
    for row in failed_rows:
        print(f" - {row['dataset']}: {row.get('error', 'unknown error')}")
print('Persistent status log:', STATE_HOME/'download_log.json')


## 2. Create a Drive-backed runtime config and run the raw source audit

The tracked repository config is not modified. The runtime config points to the Drive datasets and stores the manifest, inspection cache, embedding cache and reports in Drive. Raw audit counts are not treated as final usable training counts.

In [ ]:
RUNTIME_CFG['manifest'] = str(REPORT_HOME/'manifest_v3.csv')
RUNTIME_CFG['report_dir'] = str(REPORT_HOME/'preparation')
RUNTIME_CFG['inspect_cache'] = str(STATE_HOME/'inspect_v3.json')
RUNTIME_CFG['dedupe']['embedding']['cache_file'] = str(STATE_HOME/'embeddings_effb0_v3.npz')
# Keep resized JPEGs on Colab's fast local disk; they are reproducible from Drive source images.
RUNTIME_CFG['cache']['dir'] = '/content/smartbite_v3_resized_cache'
RUNTIME_CFG_PATH = REPO/'configs/data_v3_colab.yaml'
with RUNTIME_CFG_PATH.open('w',encoding='utf-8') as f: yaml.safe_dump(RUNTIME_CFG,f,sort_keys=False,allow_unicode=True)

os.chdir(REPO)
sys.path.insert(0,str(REPO))
try:
    run([sys.executable,'tools/audit_v3_sources.py','--profile','local','--config',str(RUNTIME_CFG_PATH),
         '--out-dir',str(REPORT_HOME/'raw_audit')])
except Exception as exc:
    error_text = f'{type(exc).__name__}: {exc}'
    (REPORT_HOME/'raw_audit_error.txt').write_text(error_text, encoding='utf-8')
    print('RAW AUDIT FAILED; pipeline will continue:', error_text)

raw_report = REPORT_HOME/'raw_audit/source_audit.md'
print('\n===== RAW SOURCE AUDIT =====\n')
print(raw_report.read_text(encoding='utf-8') if raw_report.exists() else 'Raw audit report was not created.')
with (REPORT_HOME/'download_log.json').open('w',encoding='utf-8') as f: json.dump(DOWNLOAD_LOG,f,indent=2)


## 3. Validate code and build the deduplicated manifest

This is the expensive stage. The inspection cache and EfficientNet embedding cache live in Drive, so a rerun can reuse completed work. Resized JPEGs are cached on Colab local storage for speed. If Colab resets, only that reproducible resize cache is rebuilt.

In [ ]:
manifest = Path(RUNTIME_CFG['manifest'])
prep_dir = Path(RUNTIME_CFG['report_dir'])
try:
    # Run focused preparation/checkpoint tests before touching the dataset.
    run([sys.executable,'-m','pytest','-q','tests/test_prepare.py','tests/test_v3_checkpoint_metadata.py'],cwd=REPO)

    # Confirm at least one usable dataset root exists; missing Khana is expected unless privately supplied.
    from smartbite_cv.prepare import dataset_roots
    roots = dataset_roots(RUNTIME_CFG,'local')
    print('Dataset roots available:', sorted(roots))
    if not roots:
        raise RuntimeError('No configured dataset roots were found. Check download_log.json and the raw source audit before continuing.')

    prep_cmd = [sys.executable,'-m','smartbite_cv.prepare',
                '--config',str(RUNTIME_CFG_PATH),
                '--classes','configs/classes_v3.yaml',
                '--profile','local','--workers','2','--cache',
                '--cache-dir','/content/smartbite_v3_resized_cache']
    run(prep_cmd,cwd=REPO)

    manifest = Path(RUNTIME_CFG['manifest'])
    prep_dir = Path(RUNTIME_CFG['report_dir'])
    required = [manifest, prep_dir/'prep_report.md', prep_dir/'prep_stats.json']
    missing = [str(p) for p in required if not p.exists()]
    if missing: raise RuntimeError('Preparation did not produce required outputs: '+', '.join(missing))
    print('\n===== PREPARATION REPORT =====\n')
    print((prep_dir/'prep_report.md').read_text(encoding='utf-8'))

except Exception as exc:
    error_text = f'{type(exc).__name__}: {exc}'
    prep_dir.mkdir(parents=True, exist_ok=True)
    (prep_dir/'preparation_error.txt').write_text(error_text, encoding='utf-8')
    print('PREPARATION FAILED; audit outputs will still be packaged:', error_text)


## 4. Package the audit and preparation results

Upload the ZIP to the ChatGPT conversation for review. It contains the raw audit, final manifest, preparation report, statistics, duplicate-pair audit (if generated) and download log.

In [ ]:
ZIP_PATH = REPORT_HOME/'smartbite_v3_pipeline_results.zip'
files_to_pack = [
    REPORT_HOME/'download_log.json',
    REPORT_HOME/'raw_audit/source_audit.csv',
    REPORT_HOME/'raw_audit/source_audit.md',
    manifest,
    prep_dir/'prep_report.md',
    prep_dir/'prep_stats.json',
    prep_dir/'embedding_pairs.csv',
]
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files_to_pack:
        if p.exists(): z.write(p,arcname=str(p.relative_to(REPORT_HOME)))
print('RESULT ZIP:',ZIP_PATH)
print('ZIP size (MB):',round(ZIP_PATH.stat().st_size/1024/1024,2))
print('Persistent source datasets:',DATA_HOME)
print('Persistent reports:',REPORT_HOME)


## 5. Optional training — disabled until data review is approved

Do not train just because preparation completed. First inspect the per-class train/validation/test counts, dropped classes, label-conflict clusters, duplicate-pair CSV and representative contact sheets. Once the data is approved, set `RUN_TRAINING = True` and run this cell. It trains, evaluates and exports the baseline, then copies outputs to Drive.

In [ ]:
RUN_TRAINING = False  # Change to True only after reviewing and approving the preparation results.
if not RUN_TRAINING:
    print('Training intentionally skipped. Review and approve the ZIP first.')
else:
    run([sys.executable,'-m','pytest','-q'],cwd=REPO)
    run([sys.executable,'-m','smartbite_cv.train',
         '--config','configs/train/base_v3.yaml',
         '--config','configs/train/efficientnet_b0_v3.yaml',
         '--set',f'data.config={RUNTIME_CFG_PATH}',
         '--profile','local'],cwd=REPO)
    run([sys.executable,'-m','smartbite_cv.evaluate','--run','runs/efficientnet_b0_v3','--profile','local'],cwd=REPO)
    run([sys.executable,'-m','smartbite_cv.export','--run','runs/efficientnet_b0_v3',
         '--out','artifacts/efficientnet_b0_v3'],cwd=REPO)
    backup = DRIVE_ROOT/'model_outputs'
    backup.mkdir(parents=True,exist_ok=True)
    for name in ('runs/efficientnet_b0_v3','artifacts/efficientnet_b0_v3'):
        src = REPO/name
        if src.exists():
            dst = backup/src.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(src,dst)
    print('Training, evaluation and export outputs copied to:',backup)


## Recovery behavior

- If Colab resets: reopen this notebook and rerun from cell 0. Dataset downloads are reused from Drive when their markers exist.
- If a Kaggle source fails: inspect `reports/download_log.json`; fix the Kaggle slug/permission or source marker, then rerun cell 1 onward.
- If preparation is interrupted: rerun from cell 0. The inspection and embedding caches in Drive allow unchanged files to be reused.
- Do not delete `v3_pipeline/datasets` or `v3_pipeline/state` in Drive; they contain the persistent source data and resumable caches.
- No notebook can guarantee external Kaggle availability, account permissions, Drive quota or Colab runtime uptime. This pipeline makes those failures explicit and preserves completed work.